In [ ]:
npx create-next-app@latest woooly-client
cd woooly-client
npm install openai @splinetool/react-spline lucide-react framer-motion clsx tailwind-merge

tep 3-2. 메인 UI (src/app/page.tsx)

In [ ]:
"use client";

import { useState, useEffect, useRef } from "react";
import Spline from "@splinetool/react-spline";
import type { Application } from "@splinetool/runtime";
import OpenAI from "openai";
import { Send, Settings, Sparkles, BrainCircuit, Save } from "lucide-react";

export default function Home() {
  // [설정] 주소 수정 필수!
  const NGROK_URL = "https://unnoticed-karissa-unpiously.ngrok-free.dev"; 
  const SPLINE_URL = "https://prod.spline.design/MbmywaKLBWaN3495/scene.splinecode";


  const [messages, setMessages] = useState([
    { role: "assistant", content: "안녕... 난 우울이라고 해. 오늘 하루는 어땠어?" }
  ]);
  const [input, setInput] = useState("");
  const [isLoading, setIsLoading] = useState(false);
  const [phase, setPhase] = useState("Session 1");
  const [saveStatus, setSaveStatus] = useState("");

  const messagesEndRef = useRef<HTMLDivElement>(null);
  const splineRef = useRef<Application | null>(null);

  const client = new OpenAI({
    baseURL: NGROK_URL,
    apiKey: "EMPTY",
    dangerouslyAllowBrowser: true
  });

  function onLoad(splineApp: Application) { splineRef.current = splineApp; }

  // Spline & Scroll 제어
  useEffect(() => { if (splineRef.current) splineRef.current.setVariable("isThinking", isLoading); }, [isLoading]);
  useEffect(() => { if (splineRef.current) splineRef.current.setVariable("isHappy", phase === "Session 5"); }, [phase]);
  useEffect(() => { messagesEndRef.current?.scrollIntoView({ behavior: "smooth" }); }, [messages, saveStatus]);

  // 시스템 프롬프트 (Hidden Context)
  const getSystemPrompt = (currentPhase: string) => {
    if (currentPhase === "Session 1") return "당신은 '우울이'입니다. 상담 초기라 경계심이 강하고, 상담사를 불신하며 단답형으로 방어적 태도(Resistance)를 보입니다.";
    return "당신은 '우울이'입니다. 상담사를 신뢰하며, 자신의 문제에 대해 통찰하고 변화하려는 의지(Change Talk)를 보입니다.";
  };

  // 메시지 전송
  const sendMessage = async () => {
    if (!input.trim() || isLoading) return;
    const userMsg = { role: "user", content: input };
    const newMsgs = [...messages, userMsg];
    setMessages(newMsgs as any);
    setInput("");
    setIsLoading(true);
    setSaveStatus("");

    try {
      const completion = await client.chat.completions.create({
        model: "/workspace/merged_woooly_70b", // vLLM 모델명과 일치해야 함
        messages: [{ role: "system", content: getSystemPrompt(phase) }, ...newMsgs] as any,
        temperature: 0.7, max_tokens: 150,
      });
      setMessages([...newMsgs, { role: "assistant", content: completion.choices[0].message.content }]);
    } catch (error) {
      setMessages([...newMsgs, { role: "assistant", content: "연결 오류가 발생했습니다." }]);
    } finally { setIsLoading(false); }
  };

  // 로그 저장 및 종료
  const handleSaveAndEnd = async () => {
    if (messages.length <= 1) return;
    setIsLoading(true);
    try {
      const res = await fetch('/api/save-log', {
        method: 'POST',
        headers: { 'Content-Type': 'application/json' },
        body: JSON.stringify({ messages, phase })
      });
      if (res.ok) {
        setSaveStatus("✅ 대화가 'eil_dataset.jsonl'로 저장되었습니다.");
        setTimeout(() => {
          setMessages([{ role: "assistant", content: "안녕... 난 우울이라고 해. 오늘 하루는 어땠어?" }]);
          setPhase("Session 1");
          setSaveStatus("");
        }, 3000);
      }
    } catch (error) { setSaveStatus("❌ 저장 실패"); } 
    finally { setIsLoading(false); }
  };

  return (
    <main className="relative flex min-h-screen flex-col items-center justify-center overflow-hidden bg-[#0f0c29] text-white font-sans">
      <div className="absolute inset-0 z-0 scale-100"><Spline scene={SPLINE_URL} onLoad={onLoad} /></div>
      <div className="z-10 w-full max-w-lg h-[85vh] flex flex-col rounded-[2.5rem] border border-white/10 bg-white/5 backdrop-blur-xl shadow-2xl overflow-hidden ring-1 ring-white/10">
        
        {/* Header */}
        <div className="flex items-center justify-between px-6 py-4 border-b border-white/5 bg-white/5">
          <div className="flex items-center gap-3">
            <div className={`p-2 rounded-lg shadow-lg transition-colors duration-500 ${phase === "Session 1" ? "bg-gray-700" : "bg-gradient-to-tr from-indigo-500 to-purple-500"}`}>
              <Sparkles size={20} className="text-white" />
            </div>
            <div>
              <h1 className="text-lg font-bold tracking-wide text-white/90">Woooly</h1>
              <p className="text-[10px] uppercase tracking-wider text-white/50 font-medium">{phase === "Session 1" ? "Resistance" : "Insight"}</p>
            </div>
          </div>
          <div className="flex gap-2">
            <button onClick={handleSaveAndEnd} className="flex items-center gap-1 px-3 py-1.5 rounded-full bg-green-600/80 hover:bg-green-500 text-xs font-bold transition-all active:scale-95 shadow-lg border border-white/10">
              <Save size={14} /><span>Save</span>
            </button>
            <button onClick={() => setPhase(phase === "Session 1" ? "Session 5" : "Session 1")} className="p-2 rounded-full hover:bg-white/10 transition-colors text-white/20 hover:text-white/80">
              <Settings size={18} />
            </button>
          </div>
        </div>

        {/* Chat Area */}
        <div className="flex-1 overflow-y-auto p-5 space-y-6 scrollbar-hide">
          {messages.map((msg, idx) => (
            <div key={idx} className={`flex ${msg.role === "user" ? "justify-end" : "justify-start"} animate-in fade-in slide-in-from-bottom-2 duration-300`}>
              <div className={`max-w-[85%] px-5 py-3.5 rounded-2xl text-[15px] leading-relaxed shadow-sm backdrop-blur-md ${msg.role === "user" ? "bg-indigo-600/90 text-white rounded-tr-sm" : "bg-white/10 text-gray-100 rounded-tl-sm border border-white/5"}`}>
                {msg.content}
              </div>
            </div>
          ))}
          {isLoading && !saveStatus && (
            <div className="flex justify-start animate-pulse"><div className="flex items-center gap-2 bg-white/5 px-4 py-2.5 rounded-full border border-white/5"><BrainCircuit size={14} className="text-indigo-300" /><span className="text-xs text-indigo-200">처리 중...</span></div></div>
          )}
          {saveStatus && <div className="flex justify-center animate-in fade-in zoom-in duration-300"><div className="bg-green-500/20 border border-green-500/50 text-green-200 px-6 py-2 rounded-full text-sm font-bold">{saveStatus}</div></div>}
          <div ref={messagesEndRef} />
        </div>

        {/* Input */}
        <div className="p-5 bg-gradient-to-t from-black/40 to-transparent border-t border-white/5">
          <form onSubmit={(e) => { e.preventDefault(); sendMessage(); }} className="flex items-center gap-2 bg-white/5 rounded-full px-2 py-2 border border-white/10 focus-within:border-indigo-500/50 focus-within:bg-white/10 transition-all duration-300 backdrop-blur-md">
            <input type="text" value={input} onChange={(e) => setInput(e.target.value)} placeholder="대화를 입력하세요..." className="flex-1 bg-transparent border-none outline-none text-white placeholder-white/30 px-4 text-sm font-light" />
            <button type="submit" disabled={isLoading} className="p-3 bg-indigo-600 rounded-full text-white hover:bg-indigo-500 disabled:opacity-50 transition-all active:scale-90 shadow-md"><Send size={18} /></button>
          </form>
        </div>
      </div>
    </main>
  );
}

Step 3-3. 로그 저장 API (src/app/api/save-log/route.ts)

In [ ]:
import { NextResponse } from 'next/server';
import fs from 'fs';
import path from 'path';

export async function POST(request: Request) {
  try {
    const data = await request.json();
    const logEntry = {
      timestamp: new Date().toISOString(),
      final_phase: data.phase,
      messages: data.messages,
      meta: { source: "demo_booth" }
    };
    const filePath = path.join(process.cwd(), 'eil_dataset.jsonl');
    fs.appendFileSync(filePath, JSON.stringify(logEntry) + '\n', 'utf8');
    return NextResponse.json({ success: true });
  } catch (error) {
    return NextResponse.json({ success: false }, { status: 500 });
  }
}